# 01 — Data Loading & Initial Inspection

**Goal of this notebook:** load the raw dataset and understand its structure, data types, and quality issues *before* touching anything. No cleaning happens here — only observation. Every issue noted below gets fixed in `2_dataCleaning.ipynb`.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Use a relative path (../data/raw/...) instead of a hardcoded drive path —
# this makes the notebook runnable on any machine, including after cloning from GitHub
df = pd.read_csv('../data/raw/healthcare_dataset.csv')
df.shape
# (rows, columns) — always the first thing to check after loading any dataset

(56542, 15)

In [2]:
df.head()
# Visual sanity check — confirms columns loaded into the right places,
# no shifted columns or parsing errors

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,MaTTHEw madDEN,58,Female,B+,Obesity,2024-02-05,Heather Ho,Burns-Dominguez,Cigna,9925.15,336,Elective,2024-02-13,Penicillin,Abnormal
1,tAnNEr rAYMOND,26,Female,B-,Injury,2022-10-05,Michael Obrien,Inc Scott,UnitedHealthcare,6332.10,319,Emergency,2022-10-08,Lipitor,Normal
2,MicHaEl daViS,27,Female,B-,Obesity,2020-12-07,Willie Smith,PLC Jenkins,Aetna,7580.40,140,Elective,2020-12-13,Paracetamol,Normal
3,mARc fRAnCIS,21,Male,B+,Asthma,2023-05-01,William Nelson,"Cisneros Short, and Taylor",Cigna,2418.32,374,Emergency,2023-05-03,Paracetamol,Normal
4,taRA Koch,21,Male,A+,Injury,2024-02-05,Nancy Ruiz,Erickson-Martinez,Cigna,10680.76,367,Emergency,2024-02-10,Lipitor,Abnormal


In [3]:
df.info()
# Key things flagged from this output:
# - 'Date of Admission' and 'Discharge Date' are dtype=object (text), not datetime
#   This means we currently CANNOT do date math (e.g. days stayed) until fixed
# - All other dtypes look appropriate for now

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56542 entries, 0 to 56541
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                56542 non-null  object 
 1   Age                 56542 non-null  int64  
 2   Gender              56542 non-null  object 
 3   Blood Type          56542 non-null  object 
 4   Medical Condition   56542 non-null  object 
 5   Date of Admission   56542 non-null  object 
 6   Doctor              56542 non-null  object 
 7   Hospital            56542 non-null  object 
 8   Insurance Provider  56542 non-null  object 
 9   Billing Amount      55828 non-null  float64
 10  Room Number         56542 non-null  int64  
 11  Admission Type      56542 non-null  object 
 12  Discharge Date      56542 non-null  object 
 13  Medication          56542 non-null  object 
 14  Test Results        56542 non-null  object 
dtypes: float64(1), int64(2), object(12)
memory usage: 6.5

In [ ]:
df.describe()
# Key thing flagged from this output:
# - 'Billing Amount' min is negative — a hospital bill can never be negative,
# - 'Age' min is negative — a patient can never be negative years old,also max 'Age' is 200 which is highly unlikely for a patient,
#   so this confirms a real data quality issue, not just a modeling assumption

,Age,Billing Amount,Room Number
count,56542.000000,5.582800e+04,56542.000000
mean,46.379435,8.668134e+04,301.199162
std,22.491154,8.091984e+05,115.241562
min,-15.000000,-8.500500e+03,101.000000
25%,29.000000,5.342140e+03,202.000000
50%,46.000000,8.322560e+03,302.000000
75%,61.000000,1.249018e+04,401.000000
max,200.000000,1.000000e+07,500.000000


In [5]:
df.isnull().sum()
# Checks every column for missing values — result here showed 0 missing values,
# so no imputation will be needed for this dataset

Name                    0
Age                     0
Gender                  0
Blood Type              0
Medical Condition       0
Date of Admission       0
Doctor                  0
Hospital                0
Insurance Provider      0
Billing Amount        714
Room Number             0
Admission Type          0
Discharge Date          0
Medication              0
Test Results            0
dtype: int64

In [6]:
print(f'Duplicated rows: {df.duplicated().sum()}')
# .duplicated() flags rows that are an exact copy of an earlier row
# Even one duplicate patient record would inflate counts and skew averages

Duplicated rows: 150


In [7]:
# Checking cardinality (number of unique values) of key categorical columns
# This tells us which columns are safe to group by, and which are too granular
# to reveal any pattern (e.g. if Hospital has near-1-to-1 cardinality with patients,
# grouping by Hospital won't tell us anything meaningful)

print('No. of unique values in Medical Condition:', df['Medical Condition'].nunique())
print('No. of unique values in Hospital:', df['Hospital'].nunique())
print('No. of unique values in Gender:', df['Gender'].nunique())
print('No. of unique values in Insurance Provider:', df['Insurance Provider'].nunique())
print('No. of unique values in Admission Type:', df['Admission Type'].nunique())
print('No. of unique values in Medication:', df['Medication'].nunique())

No. of unique values in Medical Condition: 7
No. of unique values in Hospital: 39352
No. of unique values in Gender: 2
No. of unique values in Insurance Provider: 5
No. of unique values in Admission Type: 3
No. of unique values in Medication: 5


In [8]:
# Quick cross-tabulation to preview whether Medication has any relationship
# with Medical Condition — useful to know before deciding which charts are worth building later
pd.crosstab(df['Medical Condition'], df['Medication'])

Medication,Aspirin,Ibuprofen,Lipitor,Paracetamol,Penicillin
Medical Condition,,,,,
Arthritis,1250,1383,1374,1369,1363
Asthma,2334,2205,2346,2250,2231
Cancer,1887,1828,1825,1840,1825
Diabetes,1402,1527,1425,1460,1380
Hypertension,625,608,632,587,672
Injury,1420,1357,1380,1382,1407
Obesity,2389,2424,2388,2383,2384


## Summary of issues found (to be fixed in notebook 2)

- `Date of Admission` and `Discharge Date` are stored as text, not datetime
- `Billing Amount` contains negative values (invalid for a hospital charge)
- `Age` column have very vast range from -ve no. to 200 some.
- No missing values and no duplicates confirmed yet at a glance — verified more rigorously next notebook
- Text casing in categorical columns not yet checked for consistency — addressed in notebook 2